# DPO Training on `with_refusals` SFT

Trains off-policy DPO on top of the `with_refusals` SFT adapter (instead of `baseline`).
Uses the same 485 preference pairs and identical hyperparameters so results are directly comparable.

Manual DPO loop — no TRL dependency.

| Step | Task | Time |
|---|---|---|
| 1 | Setup + clone repo | ~2 min |
| 2 | DPO training on `with_refusals` | ~1 hr |

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add HuggingFace token: Add-ons → Secrets → **HF_TOKEN** (toggle On)

## Step 0: Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

## Step 1: Setup

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import gc, json, torch, shutil
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model, TaskType

BASE_MODEL_ID   = "meta-llama/Llama-3.2-1B"
REPO_URL        = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
SFT_ADAPTER_DIR = "repo/outputs/with_refusals"
DPO_PAIRS_PATH  = "repo/data/dpo_pairs.jsonl"
DPO_OUTPUT_DIR  = "/kaggle/working/outputs/dpo_with_refusals"
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print(f"SFT adapter: {SFT_ADAPTER_DIR}")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded")

In [ ]:
!git clone {REPO_URL} repo
!ls repo/

## Step 2: Load Preference Pairs

In [ ]:
PROMPT_TEMPLATE = "Human: {}\n\nAssistant:"

raw_pairs = load_jsonl(DPO_PAIRS_PATH)
preference_pairs = [{
    "prompt":   PROMPT_TEMPLATE.format(p["prompt"]),
    "chosen":   p["chosen"],
    "rejected": p["rejected"],
} for p in raw_pairs]

print(f"Loaded {len(preference_pairs)} preference pairs")
if preference_pairs:
    ex = preference_pairs[0]
    print(f"\nExample:")
    print(f"  Prompt:   {ex['prompt'][:60]}...")
    print(f"  Chosen:   {ex['chosen'][:60]}...")
    print(f"  Rejected: {ex['rejected'][:60]}...")

## Step 3: Merge `with_refusals` SFT + Attach Fresh DPO LoRA

In [ ]:
print("Merging with_refusals SFT into base weights...")
MERGED_DIR = "/kaggle/working/merged_sft_with_refusals"

merge_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float32,
    device_map={"":0},
    attn_implementation="eager",
    token=os.environ["HF_TOKEN"],
)
merge_base = PeftModel.from_pretrained(merge_base, SFT_ADAPTER_DIR)
merge_base = merge_base.merge_and_unload()
merge_base.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

del merge_base
free_gpu()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float32,
    device_map={"":0},
    attn_implementation="eager",
)

dpo_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, dpo_lora)
model.print_trainable_parameters()

## Step 4: Manual DPO Training

DPO loss: `L = -log σ(β * (log π(chosen) - log π_ref(chosen) - log π(rejected) + log π_ref(rejected)))`

Reference model = same model with adapter disabled (PEFT shortcut).

⏱ ~1 hour

In [ ]:
def get_logprobs(model, input_ids, attention_mask, labels):
    """Compute per-token log probs for the response portion (where labels != -100)."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    target = labels[:, 1:]
    mask = (target != -100).float()
    safe_target = target.clamp(min=0)
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = log_probs.gather(2, safe_target.unsqueeze(2)).squeeze(2)
    return (token_logprobs * mask).sum(dim=1)


def tokenize_pair(tokenizer, prompt, response, max_len=256):
    """Tokenize prompt+response, with labels=-100 for the prompt portion."""
    full_text = prompt + " " + response
    encoded = tokenizer(
        full_text, return_tensors="pt",
        truncation=True, max_length=max_len, padding="max_length",
    )
    prompt_enc = tokenizer(prompt, add_special_tokens=False)
    prompt_len = min(len(prompt_enc["input_ids"]), max_len)

    labels = encoded["input_ids"].clone()
    labels[0, :prompt_len] = -100
    pad_mask = encoded["attention_mask"][0] == 0
    labels[0, pad_mask] = -100

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
    }

print("Helpers defined.")

In [ ]:
NUM_EPOCHS = 2
GRAD_ACCUM = 8
LR = 5e-5
BETA = 0.1
MAX_LEN = 256
MAX_GRAD_NORM = 1.0

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)

model.train()
global_step = 0
accum_loss = 0.0

print(f"Starting DPO training: {len(preference_pairs)} pairs, "
      f"{NUM_EPOCHS} epochs, grad_accum={GRAD_ACCUM}, beta={BETA}")
print(f"Optimizer steps per epoch: ~{len(preference_pairs) // GRAD_ACCUM}")

for epoch in range(NUM_EPOCHS):
    indices = torch.randperm(len(preference_pairs)).tolist()
    for i, idx in enumerate(indices):
        pair = preference_pairs[idx]
        prompt = pair["prompt"]

        chosen_tok   = tokenize_pair(tokenizer, prompt, pair["chosen"], MAX_LEN)
        rejected_tok = tokenize_pair(tokenizer, prompt, pair["rejected"], MAX_LEN)

        device = model.device
        c_ids  = chosen_tok["input_ids"].to(device)
        c_mask = chosen_tok["attention_mask"].to(device)
        c_lbl  = chosen_tok["labels"].to(device)
        r_ids  = rejected_tok["input_ids"].to(device)
        r_mask = rejected_tok["attention_mask"].to(device)
        r_lbl  = rejected_tok["labels"].to(device)

        pi_chosen  = get_logprobs(model, c_ids, c_mask, c_lbl)
        pi_rejected = get_logprobs(model, r_ids, r_mask, r_lbl)

        with torch.no_grad():
            model.disable_adapter_layers()
            ref_chosen  = get_logprobs(model, c_ids, c_mask, c_lbl)
            ref_rejected = get_logprobs(model, r_ids, r_mask, r_lbl)
            model.enable_adapter_layers()

        logits_diff = BETA * (
            (pi_chosen - ref_chosen) - (pi_rejected - ref_rejected)
        )
        loss = -F.logsigmoid(logits_diff).mean() / GRAD_ACCUM

        loss.backward()
        accum_loss += loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            print(f"  epoch {epoch+1}/{NUM_EPOCHS}  step {global_step}  "
                  f"loss={accum_loss:.4f}  "
                  f"reward_margin={logits_diff.item()/BETA:.3f}")
            accum_loss = 0.0

    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch+1} done.")

model.save_pretrained(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)
print(f"Adapter saved to {DPO_OUTPUT_DIR}")

## Step 5: Download Output

In [ ]:
shutil.make_archive("/kaggle/working/dpo_with_refusals", "zip", DPO_OUTPUT_DIR)
print("Done! Download dpo_with_refusals.zip from the Output tab.")